# Bronze ingestion — Event log

This notebook takes the raw `working_event_log` table, checks it for obvious data
problems, parses the event date into a proper timestamp, and saves the result as
`hive_metastore.bronze.bronze_event_log`.

**Columns used below** (read from how they're used, not a documented list):
- `TAG1` — the identifier used here. There are about 764k distinct values.
- `EVDATE` — the event date and time, stored as text and parsed to a timestamp.

| rows | distinct TAG1 | first date | last date |
|---|---|---|---|
| 128,099,145 | 764,456 | 2023-07-04 | 2024-07-03 |

## 1. Setup

Load the shared helper functions and imports from the project utilities notebook.

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Utils

## 2. Load the raw data

Read the source table. The preview and schema print-out confirm the columns and types
look right before going further.

In [0]:
df = spark.read.table("hive_metastore.default.working_event_log")
display(df.limit(5))
df.printSchema()

## 3. Data checks

A few checks over the whole table before saving it. None of these change the data.

### 3.1 Size

Total rows and number of distinct `TAG1` values.

In [0]:
summary = df.agg(
    F.count("*").alias("n_rows"),
    F.countDistinct("TAG1").alias("n_ids")
)
display(summary)

### 3.2 Events per date

Count how many events fall on each `ID_EVDATE`, sorted from most to least. Useful for
spotting unusually busy days or gaps in coverage.

In [0]:
display(
    df.groupBy("ID_EVDATE")
      .count()
      .orderBy(F.desc("count"))
)

### 3.3 Empty values

Count missing values per column, then the same as a share of all rows.

In [0]:
n = df.count()

nulls = df.agg(*[
    F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
    for c in df.columns
])

nulls_pct = nulls.select(*[
    (F.col(c) / F.lit(n)).alias(c.replace("null_", "pct_null_"))
    for c in nulls.columns
])

display(nulls)
display(nulls_pct)

### 3.4 Zero values

For each column, count rows holding a plain `"0"` (ignoring spaces). As in the ARQLMED
notebook, this matches only the exact text `"0"`, not `"0.0"` or similar.

In [0]:
agg_exprs = [
    F.sum((F.trim(F.col(c)) == F.lit("0")).cast("int")).alias(c)
    for c in df.columns
]

zero_counts = df.agg(*agg_exprs)
display(zero_counts)

Show the zero counts as a share of all rows.

In [0]:
zero_pct = zero_counts.select(*[
    (F.col(c) / F.lit(n)).alias(c)
    for c in zero_counts.columns
])

display(zero_pct)

## 4. Parse the event date

Turn the `EVDATE` text into a real timestamp so it can be sorted, range-checked, and
stored correctly.


Parse `EVDATE`: trim spaces, treat empty strings as missing, then convert to a
timestamp. `try_to_timestamp` returns null instead of failing on values it can't read.
This overwrites the original text column.

In [0]:
df = df.withColumn("EVDATE", F.expr("try_to_timestamp(NULLIF(trim(EVDATE), ''))"))


Re-run the summary now that `EVDATE` is a timestamp, so the earliest and latest dates
come out correctly (these are the dates shown in the table at the top).

In [0]:
summary = df.agg(
    F.count("*").alias("n_rows"),
    F.countDistinct("TAG1").alias("n_ids"),
    F.min("EVDATE").alias("min_date"),
    F.max("EVDATE").alias("max_date"),
)
display(summary)

## 5. Save to bronze

Save the table as Delta. `overwrite` plus `overwriteSchema` means you can re-run this
cell safely.

**Target table:** `hive_metastore.bronze.bronze_event_log`

In [0]:
target_catalog = "hive_metastore"     # change this
target_schema = "bronze"
target_table = "bronze_event_log"  # change this

full_name = f"{target_catalog}.{target_schema}.{target_table}"


Save the table and show the result.

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_name)
)

display(spark.table(full_name).limit(20))
